# System research notebook

Copied verbatim from `research/systems/_template/template.ipynb` by `quantlab.systems.create_system` (DESIGN §1/§3/§10 Phase 2). Every stage cell below is a thin wrapper over `quantlab`; heavy logic lives in the library, not here. All of it reads the `SYSTEM`/`CARD` dicts built in §0 from this folder's own `hypothesis.md` -- nothing below is specific to one system, including this heading; the system's name/slug is shown once §0 runs.

## 0. Setup

In [1]:
import sys
from pathlib import Path

# Jupyter's default working directory is the notebook's own folder; nbconvert callers should
# pass resources={"metadata": {"path": <this folder>}} to ExecutePreprocessor if that isn't
# already true. hypothesis.md lives right next to this notebook.
SYSTEM_DIR = Path.cwd()

# #33: `quantlab` is not pip-installed, and a fresh kernel subprocess (nbconvert/nbclient) has no
# PYTHONPATH of its own -- walk up from this folder to whichever ancestor directory holds the
# `quantlab` package (the repo root) and put that on sys.path, so the import below resolves
# whatever the kernel's cwd or install, with no environment setup required.
_root = SYSTEM_DIR
while not (_root / "quantlab" / "__init__.py").is_file():
    if _root.parent == _root:
        raise RuntimeError(f"could not find a 'quantlab' package in any parent of {SYSTEM_DIR}")
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from quantlab import config
from quantlab.systems import read_card

CARD = read_card(SYSTEM_DIR / "hypothesis.md")
SYSTEM = {
    "book": CARD["book"],
    "issue": CARD["issue"],
    "slug": CARD["slug"],
    "name": CARD.get("name", CARD["slug"]),
    "risk_type": CARD.get("risk_type"),
    "status": CARD.get("status", "testing"),
    # Set by strategy-engineer once the instrument/timeframe is picked (S3): add `symbol:` /
    # `timeframe:` (and optionally `start:` / `end:`) to hypothesis.md's front matter.
    "symbol": CARD.get("symbol"),
    "timeframe": CARD.get("timeframe"),
    "start": CARD.get("start"),
    "end": CARD.get("end"),
    # Set by S4 (quantlab.systems.set_front_matter) the first time it resolves this system's
    # study -- read here so a later stage / a later "Run all" doesn't have to re-scan the ledger.
    "study_id": CARD.get("study_id"),
}
LEDGER_DIR = config.LEDGER_DIR
STUDIES_DIR = config.STUDIES_DIR
RESULTS_DIR = SYSTEM_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SYSTEM

{'book': 'FBS',
 'issue': 23,
 'slug': 'toy_tsmom',
 'name': 'Toy H4 time-series momentum',
 'risk_type': 'A',
 'status': 'testing',
 'symbol': 'EURUSD',
 'timeframe': 'H4',
 'start': None,
 'end': None,
 'study_id': None}

## 1. Hypothesis

In [2]:
from IPython.display import Markdown, display

display(Markdown(f"**Status:** {SYSTEM['status']}  ·  **Risk type:** {SYSTEM['risk_type']}  ·  "
                 f"**Issue:** #{SYSTEM['issue']}"))
display(Markdown(CARD.get("body", "")))

**Status:** testing  ·  **Risk type:** A  ·  **Issue:** #23

**DRY RUN — toy system, not a research candidate** (DESIGN §10 Phase 2 exit test; sandbox `research/dryrun/`)

## Toy H4 time-series momentum (`toy_tsmom`)
**Book:** FBS · **Component type:** full system
**Idea (≤200 chars):** Trade EURUSD H4 in the direction of the sign of its `lookback`-bar rate of change, entering when the sign flips; ATR stop, time exit, 1 % fixed-fraction risk.
**Mechanism:** Time-series momentum (Moskowitz, Ooi & Pedersen 2012): an asset's own past return predicts its future return, attributed to initial under-reaction to information followed by delayed over-reaction, and to hedgers/central-bank flows that trade against trend. The other side is slow-moving flow (hedgers, rebalancers, mean-reversion liquidity providers). The literature evidence is at 1–12-month horizons. At H4 (lookback about 2 days to 4 weeks) the evidence for EURUSD is weak to absent. The edge here is an extrapolation of a monthly-horizon effect to a much shorter horizon, and that is the weakest part of the card. This is acceptable for a dry run.
**Falsifiable prediction:** If H4 TSMOM exists on EURUSD, trades taken in the ROC-sign direction earn a positive mean net return. Their net Sharpe is above the random-side distribution under the same schedule (see ablation), and the gross edge per trade exceeds the round-trip spread plus swap. **Kill:** the standard §4.2 gates failed after at most 3 improvement attempts. Earlier kills: (i) S1 power check. With the expected Sharpe below, MinTRL is very likely longer than the 9-year dev window, so an S1 kill is the *expected* outcome and is itself a valid dry-run path. (ii) S3 baseline, if the gross edge per trade at prior parameters is ≤ the modelled cost per trade.
**Mechanism ablation:** Randomise the **entry side** while keeping everything else: entry timestamps (every ROC zero-cross), ATR stop rule, `hold` time exit, reversal-on-cross exit timing, and sizing. At each cross, close any open position and open a new one whose side is an independent fair coin (seeded, ≥ 1000 draws). **Contradiction:** the real system's net Sharpe is not above the 95th percentile of the random-side Sharpe distribution (one-sided p ≥ 0.05), or the real-minus-median-random Sharpe difference is ≤ 0. The side is the only thing momentum explains. If a random side does as well, any profit comes from the schedule, stop or time exit, not from momentum.
**Rules:**
- Data: EURUSD H4 Bid bars, server time (EET), dev window 2016-05 → 2025-05-14 only.
- `ROC_t = close_t / close_{t-lookback} − 1`, computed at each H4 bar close t.
- Signals at bar close t, filled at the **open of bar t+1** (engine convention):
  - **Long signal:** `ROC_{t-1} ≤ 0` and `ROC_t > 0`.
  - **Short signal:** `ROC_{t-1} ≥ 0` and `ROC_t < 0`.
  - No signal otherwise. ROC exactly 0 at t is not a cross (a zero only counts as the "≤ 0 / ≥ 0" side at t−1).
- **One position at a time.** An opposite signal while in a position closes that position and opens the new one at the same next-bar open (reverse). A signal while flat (after a stop or time exit) opens a position. A same-direction signal cannot occur while in a position (crosses alternate). If one does occur after a stop or time exit, it is taken as a new entry.
- **Stop:** fixed at entry and never trailed. Level = entry fill price ∓ `stop_mult` × ATR(14)\_t (Wilder ATR on Bid bars, value at the signal bar t). Long stop below, short stop above. Fills follow DESIGN §4.3 (short stop triggers on Ask; gap-through fills at the gap open; plus slippage).
- **Target:** none.
- **Time exit:** close at the open of bar `entry_bar + hold`. The position is exposed to exactly `hold` bars; bars are counted in bars, not calendar time, so a weekend counts as 0 bars. A reversal or a new entry starts a new clock.
- **Exit priority within a bar:** the stop, if touched intrabar, comes before a time/reversal exit at the next open.
- **Sizing (type A):** risk 1 % of current equity (nominal 100,000 USD account) on the distance |entry − stop|. Lots = floor to the broker lot step (0.01) of `0.01 × equity / (stop_distance_in_pips × pip_value_per_lot)`. Flooring keeps realised risk ≤ target. If this is below the minimum lot, skip the trade and log it. At a 100k account and a typical 30–80-pip stop this gives about 1.2–3.3 lots, so rounding error is negligible.
- Costs: bar spread column (hour-aware; about 1/6 of entries fall on the 00:00 rollover bar at ≈ 2.8 pip), swap per night at 00:00 server time (triple on the rollover day), stop slippage as per the engine's cost model.
**Risk semantics:** type **A**. Fixed-fraction 1 % risk against a `stop_mult` × ATR(14) price stop; the lot varies. No target. Exits are the stop, the time exit after `hold` bars, or reversal on an opposite ROC cross. R-multiples and risk-realisation error apply.
**Symbols:** EURUSD   **Timeframe:** H4
**Expected Sharpe (annualised, net of costs):** **0.2** (plausible range 0.0–0.3). How it was estimated: MOP 2012 report single-instrument TSMOM Sharpe ratios of the order of 0.2–0.5 gross at a 12-month lookback and 1-month hold. Currencies are among the weaker asset classes there, and those figures are for monthly horizons with near-zero relative costs. At H4 horizons the signal-to-noise is lower and costs per trade are a far larger share of the move. Net 0.2 is therefore a deliberately generous point estimate, and 0 is quite possible. **Power-check note:** at SR ≈ 0.2 over ≈ 9 years, the t-stat is ≈ 0.6. S1 should flag this as underpowered.
**Expected trades/year (per symbol):** **≈ 90** at mid-grid (lookback 60). Range ≈ 65 (lookback 120) to ≈ 220 (lookback 10). How it was estimated: EURUSD H4 has ≈ 6 bars/day × ≈ 260 days ≈ 1,560 bars/year. Under a random-walk approximation, `D_t = log close_t − log close_{t−L}` has a one-bar autocorrelation of 1 − 1/L. The zero-crossing rate per bar is therefore ≈ arccos(1 − 1/L)/π ≈ √(2/L)/π: L=10 → 0.14/bar (≈ 220/yr), L=60 → 0.058/bar (≈ 90/yr), L=120 → 0.041/bar (≈ 64/yr). Each crossing is one entry (a round-trip trade). Stop and time exits end trades early but do not add entries. Trades/year ≈ crossings/year, which is independent of `stop_mult` and `hold`. Real FX has fat tails and mild short-horizon mean reversion, which usually *increases* whipsaw crossings, so these figures are lower bounds within about ±30 %.
**Free parameters (pre-registered; reviewed by the red team at S2):** 3 parameters, full grid of 12 × 7 × 10 = **840 configurations** (all count as trials).
| name | type | range [lo, hi] (or levels) | plateau scale | economic justification (1 line) |
|---|---|---|---|---|
| lookback | int | [10, 120], grid step 10 (12 levels) | relative (r = 0.20) | 10–120 H4 bars ≈ 2 days–4 weeks, the shortest horizons at which momentum from delayed reaction to macro news is claimed; a ±20 % window is the same idea |
| stop_mult | float | [1.0, 4.0], grid step 0.5 (7 levels) | relative (r = 0.20) | below 1 ATR the stop sits inside H4 noise; above ≈ 4 ATR it is effectively no stop over a ≤ 10-day hold; the zero is meaningful (distance) |
| hold | int | [6, 60], grid step 6 (10 levels) | relative (r = 0.20) | 1–10 trading days: the persistence horizon should be comparable to the lookback, and the exit should come before the autocorrelation is expected to decay; the zero is meaningful (duration) |
  No unordered categoricals. The judge's 5 %-of-range floor (lookback 5.5 bars, stop_mult 0.15, hold 2.7 bars) overrides r/2 at the low end of lookback and hold. Integer perturbations need a rounding rule; see the report.
**Cost sensitivity:**
- Typical EURUSD H4 ATR(14) is ≈ 15–25 pips. At mid-grid, the average holding time is about 15–30 bars, because reversals and stops cut many trades short of `hold`. The per-trade return s.d. is ≈ 20 × √20 ≈ 90 pips.
- A plausible gross TSMOM edge of 2–5 % of the per-trade s.d. is ≈ **2–4.5 pips gross per trade**.
- Costs per round trip:
  - **Spread:** ≈ 0.8–1.0 pip (about 1/6 of entries on the rollover bar at ≈ 2.8 pip, so ≈ 1.2 pip on average).
  - **Stop slippage:** on stopped trades.
  - **Swap:** an average hold of about 3–5 nights with one triple night, at |swap| ≈ 0.3–1.0 pip/night, gives ≈ 1–5 pips on the paying side. Long EURUSD paid negative carry for most of 2016–2022. Shorts earned partly back, less the broker markup.
- Net cost is ≈ **1.5–4 pips/trade** against 2–4.5 pips gross. The headroom is **thin to negative**: a break-even spread multiple of about 1–2× is expected. The 1.5× spread + 1-pip slippage stress (§4.2) will likely fail even if the baseline is positive. At lookback 10 (≈ 220 trades/yr) costs dominate outright.
**Null / benchmark for component tests:** This is a full system, so the §4.2 entry-signal null applies. Random entries at the **same timestamps**, with random side (fair coin, seeded, ≥ 1000 draws), each held for the **same number of bars as the corresponding real trade**, with the same sizing and costs. This preserves the entry schedule and the holding-time distribution. The system must beat this null's 95th percentile on net Sharpe. A secondary benchmark is buy-and-hold / short-and-hold EURUSD over the dev window, to show that a net-long or net-short drift is not driving the result.
**Sources:**
- Moskowitz, T. J., Ooi, Y. H., & Pedersen, L. H. (2012). *Time series momentum.* Journal of Financial Economics, 104(2), 228–250. DOI: 10.1016/j.jfineco.2011.11.003. The strategy definition (sign of the past-12-month excess return, volatility-scaled position) and the per-instrument / per-asset-class results (incl. currencies) are in the time-series-momentum trading-strategy section. *Page/section not re-read for this dry run; the lead should verify the exact section before publishing, as the Scout rules require.*
- H4 horizon, ROC-cross entry, ATR stop and time exit are toy choices given by the lead for the dry run, not taken from a source.
**Provenance:** literature (toy). Pre-specified by the lead for the Phase 2 dry run; 0 mined candidates scanned. **Related prior work (not in the ledger, legacy/pre-pipeline):** `DocumentationVault/strategies/01_SMA_Crossover_ATR_Risk.md` and `04_MACD_Histogram_Momentum.md`. Both are trend/momentum signals, but neither is a ROC-sign cross with a time exit. ROC(L) > 0 is equivalent to close > close_{t−L}, a close cousin of a price/SMA cross. The red team may argue family overlap for the DSR trial count. `research/ledger/studies.jsonl` does not exist yet, so no ledger duplicates; GitHub search for "momentum" returned no hypothesis issues.

## 2. Implementation

In [3]:
import inspect
from dataclasses import asdict

from quantlab.costs import InstrumentSpec
from quantlab.strategies import get_strategy
from quantlab.systems import card_space_corners
from quantlab.testing import (
    assert_engine_causal,
    assert_no_lookahead,
    assert_strategy_source_clean,
    synthetic_bars,
)

StrategyCls = get_strategy(SYSTEM["slug"])
assert_strategy_source_clean(inspect.getfile(StrategyCls))

# Same class/Params resolution order as quantlab.evaluators.RuleEvaluator.build_strategy: a
# strategy is built from its declared prior (default) parameters here, so this audit exercises
# the real class with the card's own prior values, not an arbitrary stand-in.
_params_cls = getattr(StrategyCls, "params_cls", None) or getattr(StrategyCls, "Params", None)

# #31: audit the prior AND every corner of the card's pre-registered grid, when it declares one
# parseable this way (opt.SearchSpace itself isn't built until S4) -- {} = the prior (no override).
_param_overrides = [{}]
if _params_cls is not None:
    _param_overrides += card_space_corners(CARD)

_audit_bars = synthetic_bars(600, seed=0, timeframe="H1")
_audit_spec = InstrumentSpec(
    symbol="AUDIT", digits=5, point=1e-5, contract_size=100_000.0, tick_size=1e-5,
    volume_min=0.01, volume_step=0.01, volume_max=100.0, base_ccy="EUR", quote_ccy="USD",
    swap_mode="points", swap_long=0.0, swap_short=0.0, swap_3day=2,
    commission_per_lot_rt=0.0, stops_level=0.0, calibrated=True,
)
for _override in _param_overrides:
    if _params_cls is not None:
        _factory = lambda _o=_override: StrategyCls(_params_cls(**{**asdict(_params_cls()), **_o}))
    else:
        _factory = StrategyCls
    assert_no_lookahead(_factory, _audit_bars)
    assert_engine_causal(_factory(), _audit_bars, _audit_spec, n_checks=8, seed=1, min_history=60)

print(f"{StrategyCls.name}: look-ahead audit clean at the prior and {len(_param_overrides) - 1} "
     f"grid corner(s) (risk_type={StrategyCls.risk_type})")

toy_tsmom: look-ahead audit clean at the prior and 8 grid corner(s) (risk_type=RiskType.A)


## 3. Baseline

In [4]:
if SYSTEM.get("symbol") and SYSTEM.get("timeframe"):
    from quantlab.evaluators import RuleEvaluator, edge_breakdown
    from quantlab.systems import log_baseline

    evaluator = RuleEvaluator(
        StrategyCls, symbol=SYSTEM["symbol"], timeframe=SYSTEM["timeframe"], book=SYSTEM["book"],
        start=SYSTEM.get("start"), end=SYSTEM.get("end"),
    )
    baseline = evaluator({})   # {} = the card's prior (default) parameters, costs on
    print(evaluator.describe())
    print(baseline.metrics)                              # #24: explicit -- this is the `if`
                                                            # branch, so auto-display never fires
    edge = edge_breakdown(baseline, cost=evaluator.cost)   # #25: gross/net/spread/swap/slippage
    print(edge)                                            # per trade, trades/year, cost % gross

    # #28: an audit-trail event, not a new study row -- see quantlab.systems.log_baseline.
    _params_used = _params_cls().as_dict() if _params_cls is not None else {}
    _logged = log_baseline(
        SYSTEM["book"], SYSTEM["issue"], SYSTEM["slug"],
        params=_params_used, edge_breakdown=edge, cost_model_version=evaluator.cost_version,
    )
    if _logged is None:
        print("S3: no study in the ledger yet for this system -- baseline not logged to the "
             "ledger (re-run this cell after S4; #35, S3 is independently re-runnable), or a "
             "'baseline' event is already logged for it.")
    else:
        print(f"S3: baseline logged to the ledger under {_logged['study_id']}")
else:
    evaluator = None
    baseline = None
    edge = None
    print("S3 pending: add `symbol:` / `timeframe:` to hypothesis.md's front matter, then re-run.")

/home/douglaso/Finances/DataScienceAndTrading/quantlab/evaluators.py:409: UserWarning: no calibrated broker export for FBS/EURUSD; using an uncalibrated fallback InstrumentSpec (DESIGN §10 Phase 0). Run tools/mql5/ExportBrokerSpecs.mq5 and re-run to calibrate.
  spec = load_instrument(self.symbol, book=self.book)


{'evaluator': 'RuleEvaluator', 'strategy': 'toy_tsmom', 'symbol': 'EURUSD', 'timeframe': 'H4', 'book': 'FBS', 'start': 'None', 'end': '2025-05-15 00:00:00', 'sizing': 'fixed_fraction', 'risk_fraction': 0.01, 'lots': 1.0, 'use_m1': True, 'fixed_params': {}, 'warmup_bars': 500, 'cost_model_version': 'fbs-v2-uncalibrated'}
{'sharpe': -0.41358347960157416, 'sortino': -0.58206694925602, 'cagr': -0.03540836981012896, 'max_dd': -0.36956321150890614, 'calmar': -0.09581140304945003, 'ulcer': 0.21514863790718683, 'cvar95': -0.012001998445296218, 'skew': 0.13882135001065732, 'kurtosis': 6.582720008181439, 'longest_dd_days': 1991.0, 'mean_monthly': -0.0026983619528350255, 'worst_month': -0.07863383966187687, 'n_days': 2352.0, 'n_trades': 780.0, 'win_rate': 0.31153846153846154, 'profit_factor': 0.8556690608614688, 'expectancy_points': -30.836948474208185, 'avg_hold_bars': 9.41025641025641, 'trades_across_data_gap': 0.0, 'hold_days_max': 6.0, 'hold_days_p95': 5.0, 'hold_calendar_days_max': 10.0, 'ho

## 4. Optimisation

In [5]:
if evaluator is not None:
    import time

    from quantlab import ledger, opt
    from quantlab.systems import find_study_id, set_front_matter

    # S4 (optimization-architect). Search space = the card's pre-registered table, verbatim:
    #   lookback  int   [10, 120] step 10  (12 levels)  relative, r = 0.20
    #   stop_mult float [1.0, 4.0] step 0.5 (7 levels)  relative, r = 0.20
    #   hold      int   [6, 60]   step 6   (10 levels)  relative, r = 0.20
    # 12 x 7 x 10 = 840 configurations; method="auto" -> full grid (3 discrete params), which is
    # data-independent. ATR period (14) is fixed by the card, not searched.
    space = opt.SearchSpace((
        opt.IntParam("lookback", 10, 120, step=10, plateau_scale="relative"),
        opt.FloatParam("stop_mult", 1.0, 4.0, step=0.5, plateau_scale="relative"),
        opt.IntParam("hold", 6, 60, step=6, plateau_scale="relative"),
    ))
    PLATEAU_RADIUS = 0.20          # card: "relative (r = 0.20)" on every row
    # S2->S4 hand-off check: the code's space carries exactly the card's table.
    _card_table = {
        "lookback": dict(kind="int", low=10, high=120, step=10, plateau_scale="relative", n=12),
        "stop_mult": dict(kind="float", low=1.0, high=4.0, step=0.5, plateau_scale="relative", n=7),
        "hold": dict(kind="int", low=6, high=60, step=6, plateau_scale="relative", n=10),
    }
    for _p in space.params:
        _c = _card_table[_p.name]
        assert (_p.kind, _p.low, _p.high, _p.step, _p.plateau_scale, len(_p.grid_values())) == \
            (_c["kind"], _c["low"], _c["high"], _c["step"], _c["plateau_scale"], _c["n"]), _p
    assert set(space.names) == set(_card_table) and space.grid_size() == 840

    # Fixed before looking at results:
    #  * objective: default block_sharpe (4 contiguous blocks, mean - 0.5*std), net of costs;
    #    min_trades = 200 on the full ~9-y dev window (>= 160 in a CPCV train set = 8/10 of rows,
    #    i.e. >= ~50 trades per free parameter per fold; the card expects >= ~65 trades/yr, so this
    #    only removes pathological configurations);
    #  * CPCV(10 groups, k=2), purge/embargo = auto (max holding period of ok trials, capped at 1/4 group);
    #  * anchored walk-forward, quarterly re-fit, 2-year minimum training window;
    #  * selection: plateau (grid Chebyshev radius 1, default PlateauConfig);
    #  * evaluator from S3: M1 intrabar resolution (use_m1=True), default FBS cost model.
    STUDY_KW = dict(
        book=SYSTEM["book"], system=SYSTEM["slug"], issue=SYSTEM["issue"], attempt=1, method="auto",
        seed=0, n_jobs=8, cv=opt.CPCVConfig(n_groups=10, k_test=2),
        wfo=opt.WFOConfig(refit_every="3mo", window="anchored", min_train="2y"),
        objective=opt.Objective(kind="block_sharpe", n_blocks=4, lam=0.5), min_trades=200,
        plateau_radius=PLATEAU_RADIUS, checkpoint_every=100,
        notes="DRY RUN (DESIGN §10 Phase 2 exit test); toy system, not a research candidate",
    )

    # #35: a full "Run all" must not repeat a logged run_study -- load the ledger's own study
    # (found by id, else by a book/issue/slug ledger lookup) instead of blindly re-running it.
    study_id = SYSTEM.get("study_id") or find_study_id(SYSTEM["book"], SYSTEM["issue"], SYSTEM["slug"])
    if study_id is not None:
        study = opt.load_study(study_id)
        print(f"S4: loaded existing study {study_id} read-only (run_study skipped, #35)")
    else:
        _t0 = time.perf_counter()
        study = opt.run_study(evaluator, space, **STUDY_KW)
        study_id = study.study_id
        print(f"{study.study_id}: wall {time.perf_counter() - _t0:.1f}s, "
             f"n_trials={study.meta['n_trials']} (ok={study.meta['n_ok']}, low_trades={study.meta['n_low_trades']}, "
             f"invalid={study.meta['n_invalid']}, error={study.meta['n_error']})")
    print("selected:", study.selected_params)
    print("selection:", {k: study.selection.get(k) for k in (
        "selected_trial", "objective", "smoothed", "sharpe", "plateau_score", "n_neighbours",
        "raw_argmax_params", "raw_argmax_objective", "raw_argmax_smoothed", "raw_argmax_plateau_score")})
    print("cv:", study.meta.get("cv_scheme"), "| embargo_capped:", study.meta.get("embargo_capped"),
          "| embargo uncapped (days):", study.meta.get("embargo_days_uncapped"))
    print("wfo:", {k: v for k, v in (study.meta.get("wfo") or {}).items() if k != "drift_distances"})

    if study_id is not None and CARD.get("study_id") != study_id:
        set_front_matter(SYSTEM_DIR / "hypothesis.md", study_id=study_id)
        CARD["study_id"] = study_id
    SYSTEM["study_id"] = study_id
    study.wfo_params
else:
    space = None
    study = None
    study_id = None
    print("S4 pending: needs a baseline (S3) first.")

S4: loaded existing study fbs-0023-a1 read-only (run_study skipped, #35)
selected: {'hold': 18, 'lookback': 80, 'stop_mult': 1.0}
selection: {'selected_trial': 492, 'objective': -0.3268460609797821, 'smoothed': -0.12485929697210858, 'sharpe': -0.11775327003742973, 'plateau_score': 0.0, 'n_neighbours': 17, 'raw_argmax_params': {'hold': 12, 'lookback': 70, 'stop_mult': 1.0}, 'raw_argmax_objective': 0.12841327560653532, 'raw_argmax_smoothed': -0.2435091188124788, 'raw_argmax_plateau_score': 0.11764705882352941}
cv: CPCV(n=10,k=2,purge=14d,embargo=14d)+WFO(refit=3mo,window=anchored,min_train=2y) | embargo_capped: False | embargo uncapped (days): 14
wfo: {'scheme': 'WFO(refit=3mo,window=anchored,min_train=2y)', 'n_refits': 29, 'drift_mean': 0.08845086049885238, 'drift_max': 0.5797311051426967, 'drift_share_jumps_gt_0.25': 0.10714285714285714, 'n_distinct_selections': 11}


## 5. Validation

In [6]:
if evaluator is not None:
    import polars as pl

    from quantlab import gates, ledger, opt
    from quantlab.systems import find_study_id

    # Resolved independently of §4's own `study` variable (#35/#68: this cell must work even if
    # §4 did not run this kernel session -- only the ledger/trial-store, loaded by id, are trusted).
    study_id = SYSTEM.get("study_id") or find_study_id(SYSTEM["book"], SYSTEM["issue"], SYSTEM["slug"])
    study = opt.load_study(study_id) if study_id is not None else None

    if study is None:
        gate_report = None
        gate_table = None
        print("S5 pending: needs an optimisation study (S4) first.")
    elif ledger.study_events(study.study_id, "gates"):
        gate_report = None
        gate_table = None
        print(f"S5: {study.study_id} already has a logged gates event -- "
             "evaluate_gates(log=True) skipped (#35: a full 'Run all' must never log it twice).")
    else:
        gate_report = gates.evaluate_gates(study, evaluator, periods_per_year=evaluator.periods_per_year, log=True)
        gate_table = pl.DataFrame([
            {"gate": r.gate, "value": r.display if r.display is not None else r.value,
             "threshold": r.threshold, "status": r.status, "interpretation": r.interpretation}
            for r in gate_report.rows
        ])
    gate_table
else:
    study = None
    gate_report = None
    gate_table = None
    print("S5 pending: needs a baseline (S3) first.")

S5: fbs-0023-a1 already has a logged gates event -- evaluate_gates(log=True) skipped (#35: a full 'Run all' must never log it twice).


## 6. Red-team findings

In [7]:
FINDINGS_PATH = RESULTS_DIR / "redteam_findings.md"
if FINDINGS_PATH.exists():
    from IPython.display import Markdown, display

    display(Markdown(FINDINGS_PATH.read_text()))
else:
    print(f"S6 pending: no findings file yet at {FINDINGS_PATH}")

S6 pending: no findings file yet at /home/douglaso/Finances/DataScienceAndTrading/research/dryrun/systems/FBS/0023_toy_tsmom/results/redteam_findings.md


## 7. Report

In [8]:
if evaluator is not None:
    from quantlab import ledger, report
    from quantlab.systems import find_study_id

    # #68: resolved independently of §4/§5's own variables -- report.tear_sheet only needs a
    # study_id + evaluator, reading everything else (including the gates event) from the ledger
    # and trial store itself, so this cell works even run alone in a fresh kernel.
    study_id = SYSTEM.get("study_id") or find_study_id(SYSTEM["book"], SYSTEM["issue"], SYSTEM["slug"])
    if study_id is not None and ledger.study_events(study_id, "gates"):
        ts = report.tear_sheet(study_id, evaluator=evaluator, risk_type=SYSTEM.get("risk_type"))
        ts.write_results(RESULTS_DIR)
        report.write_card(
            ts, slug=SYSTEM["slug"], name=SYSTEM["name"], idea=CARD.get("idea", ""),
            status=SYSTEM.get("status", "testing"), issue=SYSTEM["issue"], book=SYSTEM["book"],
        )
    else:
        ts = None
        print("S7 pending: needs validation (S5, a logged gates event) first.")
else:
    ts = None
    print("S7 pending: needs a baseline (S3) first.")
ts.headline if ts is not None else None

{'study_id': 'fbs-0023-a1',
 'book': 'FBS',
 'system': 'toy_tsmom',
 'issue': 23,
 'risk_type': 'A',
 'class': 'unprofitable',
 'mean_monthly': -0.00014826601331920442,
 'mean_monthly_pct': -0.01482660133192044,
 'mean_monthly_band': (-0.0007858606244749482,
  -0.00015833777542830592,
  0.0005286777234694226),
 'gate_verdict': 'FAIL',
 'robustness': {'dsr': 0.00019700712894813049,
  'cscv_oos_loss': 0.7887334887334887,
  'oos_sharpe': -0.6825940469582321,
  'wfo_oos': 0.06735869322974737,
  'wfo_oos_recent': -0.07403172677579248,
  'plateau': 0.0,
  'cost_stress_sharpe': -0.9174275932628458,
  'positive_years': 0.6,
  'max_year_share': None,
  'holdout_status': 'not unlocked',
  'pbo': 0.5312354312354313},
 'risk_profile': 'Trend-like',
 'key_risks': ['Failed gate(s): Deflated Sharpe probability, CSCV OOS loss probability, OOS Sharpe (CPCV median), Walk-forward procedure OOS, Cost-stress Sharpe, Parameter plateau, Max single-year PnL share, Trade count vs MinTRL.',
  'Holdout not unloc

## 8. Holdout

In [9]:
# Status only. The holdout is unlocked once, by the user, via `/unlock-holdout <system>` -- never
# from this notebook. This cell never calls quantlab.data with include_holdout=True.
print(f"S8 holdout: system status = {SYSTEM.get('status')!r} (book {SYSTEM['book']}). "
     "Run `/unlock-holdout` yourself when checkpoint B is reached (DESIGN §3, §4.4).")

S8 holdout: system status = 'testing' (book FBS). Run `/unlock-holdout` yourself when checkpoint B is reached (DESIGN §3, §4.4).
